# Slices y sesgo: el promedio miente

**Facsímil 8 · La ciencia de los datos** — capítulo 5 (slices, sesgos y decisión algorítmica).

Un acierto global estupendo puede esconder que tu modelo funciona **fatal** con un grupo concreto. El
número global —ese 90% que tranquiliza— está dominado por la mayoría, y puede tapar que el sistema falla
justo con quien menos representado está. En este cuaderno coges un modelo con buen acierto global, lo
**partes por grupos** (*slices*) y descubres el sesgo que el promedio ocultaba. Si decides sobre
personas —créditos, selección, diagnóstico—, el promedio no te exime: tienes que mirar **a quién estás
fallando**.

### Qué vas a aprender
- Por qué un **acierto global alto** puede convivir con un fallo grave en un subgrupo.
- A medir el rendimiento **por *slices*** (por grupo, segmento o tipo de caso).
- Que cuanto más **minoritario** es un grupo, más invisible es para el promedio.
- A pensar en **mitigaciones** (reequilibrar) y en sus compromisos.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
np.random.seed(0)

# Dos grupos. El A es mayoria (85%) y tiene una relacion clara senal->etiqueta.
# El B es minoria (15%) y su patron es DISTINTO: el modelo, dominado por A, le va mal.
def genera(n, grupo):
    x = np.random.normal(0, 1, (n, 3))
    if grupo == "A":
        y = (x[:,0] + x[:,1] > 0).astype(int)
    else:
        y = (x[:,2] - x[:,0] > 0).astype(int)   # otra regla
    return x, y, np.array([grupo]*n)

xa, ya, ga = genera(1700, "A")
xb, yb, gb = genera(300, "B")
X = np.vstack([xa, xb]); y = np.concatenate([ya, yb]); g = np.concatenate([ga, gb])
idx = np.random.permutation(len(y)); X, y, g = X[idx], y[idx], g[idx]
corte = 1400
Xtr, ytr = X[:corte], y[:corte]
Xte, yte, gte = X[corte:], y[corte:], g[corte:]
print(f"Entrenamiento: {corte} casos (85% grupo A, 15% grupo B, con reglas distintas).")


Entrenamiento: 1400 casos (85% grupo A, 15% grupo B, con reglas distintas).


## 1. El número que tranquiliza

Entrenamos un modelo y miramos su acierto global. Sale alto, el típico número que aparece en una
presentación y hace asentir a todo el mundo. Pero un número global es un **promedio**, y los promedios
esconden lo que pasa dentro.


In [2]:
modelo = LogisticRegression().fit(Xtr, ytr)
pred = modelo.predict(Xte)
acc_global = (pred == yte).mean()
print(f"Acierto GLOBAL: {acc_global:.3f}   <- 'el modelo va bien', diria cualquiera")


Acierto GLOBAL: 0.832   <- 'el modelo va bien', diria cualquiera


## 2. Ahora míralo por grupos

El mismo modelo, las mismas predicciones, pero separando el acierto del grupo A y del grupo B. Aquí es
donde el promedio confiesa lo que escondía. Calcular métricas **por *slice*** es la herramienta más
simple y más reveladora de la ciencia de datos responsable.


In [3]:
print(f"{'grupo':>6} | {'casos':>6} | {'acierto':>8}")
for grupo in ["A", "B"]:
    m = gte == grupo
    print(f"{grupo:>6} | {m.sum():>6} | {(pred[m]==yte[m]).mean():>8.3f}")
print("\nEl global se parecia al del grupo MAYORITARIO. El grupo B, minoria, queda mucho peor servido.")


 grupo |  casos |  acierto
     A |    510 |    0.916
     B |     90 |    0.356

El global se parecia al del grupo MAYORITARIO. El grupo B, minoria, queda mucho peor servido.


**Esto es lo que el promedio tapa.** El acierto global se parece sospechosamente al del grupo
mayoritario, porque son la mayoría de los casos y mandan en la media. El grupo B, pequeño, casi no mueve
el número global... pero recibe un servicio mucho peor. Si ese grupo son personas, acabas de descubrir
un sesgo que ninguna métrica global te iba a enseñar.


## 3. Por qué pasa: la minoría no mueve el promedio

Cuanto más pequeño es un grupo, menos pesa su (mal) rendimiento en el número global, y más invisible se
vuelve. Lo medimos: hacemos el grupo B cada vez más minoritario y vemos cómo su mal acierto apenas mueve
el global, aunque para esas personas el sistema siga siendo malo.


In [4]:
def experimento(frac_B):
    nB = int(2000 * frac_B); nA = 2000 - nB
    xa, ya, _ = genera(nA, "A"); xb, yb, _ = genera(nB, "B")
    X = np.vstack([xa, xb]); y = np.concatenate([ya, yb]); gg = np.array(["A"]*nA + ["B"]*nB)
    idx = np.random.permutation(len(y)); X, y, gg = X[idx], y[idx], gg[idx]
    c = 1600; m = LogisticRegression().fit(X[:c], y[:c])
    pr = m.predict(X[c:]); yt, gt = y[c:], gg[c:]
    glob = (pr==yt).mean()
    accB = (pr[gt=="B"]==yt[gt=="B"]).mean() if (gt=="B").sum() else float("nan")
    return glob, accB

print("% grupo B | acierto GLOBAL | acierto del grupo B")
for frac in [0.30, 0.15, 0.05]:
    glob, accB = experimento(frac)
    print(f"   {frac*100:>2.0f}%    |     {glob:.3f}     |       {accB:.3f}")
print("\nCuanto mas pequeno es B, MENOS afecta su mal acierto al global: mas invisible para el promedio.")


% grupo B | acierto GLOBAL | acierto del grupo B
   30%    |     0.748     |       0.492
   15%    |     0.870     |       0.516
    5%    |     0.945     |       0.393

Cuanto mas pequeno es B, MENOS afecta su mal acierto al global: mas invisible para el promedio.


## 4. Mitigar: reequilibrar (y su precio)

Una vez visto el sesgo, ¿se puede corregir? Una opción: dar **más peso** al grupo minoritario al
entrenar, para que el modelo no lo ignore. Casi siempre hay una **balanza**: mejorar el grupo B puede
empeorar un poco el A. Lo importante es que esa decisión sea **explícita**, no un accidente del promedio.


In [5]:
# damos mas peso a los casos del grupo B durante el entrenamiento
peso = np.where(g[:corte] == "B", 5.0, 1.0)
modelo_eq = LogisticRegression().fit(Xtr, ytr, sample_weight=peso)
pred_eq = modelo_eq.predict(Xte)
print(f"{'grupo':>6} | acierto SIN reequilibrar | acierto CON reequilibrar")
for grupo in ["A", "B"]:
    m = gte == grupo
    a0 = (pred[m]==yte[m]).mean(); a1 = (pred_eq[m]==yte[m]).mean()
    print(f"{grupo:>6} |          {a0:.3f}          |          {a1:.3f}")
print("\nReequilibrar suele subir al grupo B a costa de algo del A. La balanza se decide a proposito.")


 grupo | acierto SIN reequilibrar | acierto CON reequilibrar
     A |          0.916          |          0.720
     B |          0.356          |          0.656

Reequilibrar suele subir al grupo B a costa de algo del A. La balanza se decide a proposito.


## 5. Pruébalo tú

1. **Otras métricas por *slice*:** mira los falsos positivos y falsos negativos por grupo, no solo el
   acierto. El sesgo puede estar en *cómo* falla, no solo en cuánto.
2. **Más grupos:** añade un grupo C con otra regla. ¿El promedio sigue tapando a los pequeños?
3. **Ajusta el peso** del reequilibrio (2, 5, 10): ¿hasta dónde mejora B sin hundir A? Esa es la frontera
   que tú decides.
4. **Paridad:** define una métrica de equidad (p. ej. que el acierto no difiera más de X puntos entre
   grupos) y comprueba si tu modelo la cumple.


## 6. Errores comunes

- **Reportar solo el número global.** Esconde a quién estás fallando. Mide siempre por *slices*
  relevantes.
- **Olvidar los grupos pequeños.** Cuanto más minoritarios, más invisibles para el promedio, y más fácil
  ignorarlos sin querer.
- **Creer que el sesgo se arregla solo mirándolo.** No: hay que decidir una mitigación y aceptar sus
  compromisos. Pero no puedes arreglar lo que no mides.
- **Confundir acierto con justicia.** Un modelo «bueno de media» puede ser injusto para un colectivo.


## 7. Qué te llevas

- Un **acierto global alto** puede convivir con un fallo grave en un subgrupo: el promedio está dominado
  por la mayoría.
- **Medir por *slices*** (por grupo, segmento o tipo de caso) es la forma de ver a quién estás fallando,
  la pregunta que de verdad importa cuando decides sobre personas.
- El sesgo no se arregla solo mirándolo, pero **no puedes arreglar lo que no mides**, y reequilibrar tiene
  un precio que conviene decidir a propósito.

**Para seguir:** el facsímil 9 (seguridad, privacidad y gobernanza) convierte esto en obligación: medir
el impacto por grupos es parte de tratar la IA de forma responsable y auditable.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*